# Retrieval-Augmented Generation (RAG) Pipeline for TourismGPT

## Overview

This notebook implements the Retrieval-Augmented Generation (RAG) pipeline used by TourismGPT. While the fine-tuned Phi-3 Mini model provides strong language generation capabilities, the RAG pipeline enhances its responses by retrieving relevant tourism information from an external knowledge base before generating an answer.

The pipeline combines semantic search using Sentence Transformers with FAISS vector indexing to retrieve the most relevant tourism documents. These retrieved documents are supplied as context to the language model, enabling more accurate, context-aware, and reliable responses.

# TourismGPT — RAG Pipeline
Parses Wikivoyage → FAISS index → retrieval-augmented Phi-3 inference.

**Before running:** Upload `enwikivoyage-latest-pages-articles.xml.bz2` and `tourism_gpt_adapter/` folder to your Google Drive.

## Environment Setup

The required libraries are installed to support document processing, semantic embedding generation, vector indexing, and information retrieval.

These tools enable the chatbot to efficiently search large collections of tourism documents and retrieve relevant information before generating responses.

In [1]:
# CELL 1 — Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-gpu

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xexg30x2/unsloth_d129a2b99fbe4157a1cba9f5d96f8720
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xexg30x2/unsloth_d129a2b99fbe4157a1cba9f5d96f8720
  Resolved https://github.com/unslothai/unsloth.git to commit 278e9e7921a56c603a3384e1bdc8562c4e354858
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformer

In [2]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Required Libraries

This section imports all libraries required for building the Retrieval-Augmented Generation pipeline.

The imported modules support document loading, embedding generation, similarity search, vector storage, and interaction with the fine-tuned language model.

## Loading the Tourism Knowledge Base

The RAG pipeline begins by loading tourism-related documents that serve as the external knowledge source.

These documents contain information about destinations, attractions, transportation, accommodation, local culture, and travel guidance. They provide factual context that complements the knowledge learned during model fine-tuning.

In [3]:
import bz2, re, os, json, pickle
import numpy as np
import torch
from xml.etree import ElementTree as ET
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
WIKIVOYAGE_BZ2  = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/enwikivoyage-latest-pages-articles.xml.bz2"
ADAPTER_DIR     = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter"
INDEX_DIR       = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index"
DRIVE_INDEX     = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index"

# ── RAG config ─────────────────────────────────────────────────────────────
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE      = 400    # words per chunk
CHUNK_OVERLAP   = 50
TOP_K           = 3      # chunks to retrieve per query
MAX_ARTICLES    = 5000   # cap to keep index build time under 5 min

# ── Phi-3 config ───────────────────────────────────────────────────────────
MAX_SEQ_LEN     = 2048
MAX_NEW_TOKENS  = 400

print("Config ready ✓")

Config ready ✓


In [4]:
# CELL 4 — Parse Wikivoyage XML (namespace-agnostic)

_STRIP_RE = re.compile(
    r'\{\{[^}]*\}\}'
    r'|\[\[(?:[Ff]ile|[Ii]mage):[^\]]*\]\]'
    r'|\[\[(?:[Cc]ategory):[^\]]*\]\]'
    r'|<[^>]+>'
    r'|\[\[(?:[^\]|]*\|)?([^\]]*)\]\]',
    re.DOTALL
)

def clean_wikitext(raw):
    text = _STRIP_RE.sub(lambda m: m.group(1) or "", raw)
    text = re.sub(r"={2,}[^=]+=+", " ", text)
    text = re.sub(r"''+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_article(title):
    prefixes = ("Wikivoyage:", "Talk:", "User:", "Template:", "Help:",
                "File:", "MediaWiki:", "Category:", "Module:")
    return not any(title.startswith(p) for p in prefixes)

def detect_namespace(bz2_path):
    with bz2.open(bz2_path, "rb") as f:
        header = f.read(4096).decode("utf-8", errors="ignore")
    match = re.search(r'xmlns="([^"]+)"', header)
    if match:
        ns = match.group(1)
        print(f"Detected namespace: {ns}")
        return ns
    print("Namespace not found, using default")
    return "http://www.mediawiki.org/xml/export-0.10/"

def parse_wikivoyage(bz2_path, max_articles=None):
    ns = detect_namespace(bz2_path)
    articles = []
    count = 0
    print(f"Parsing {bz2_path} ...")
    with bz2.open(bz2_path, "rb") as f:
        context = ET.iterparse(f, events=("end",))
        for event, elem in context:
            tag = elem.tag.replace(f"{{{ns}}}", "")
            if tag == "page":
                title_el = elem.find(f"{{{ns}}}title")
                text_el  = elem.find(f".//{{{ns}}}revision/{{{ns}}}text")
                if title_el is not None and text_el is not None:
                    title = title_el.text or ""
                    raw   = text_el.text  or ""
                    if is_article(title) and len(raw) > 200:
                        clean = clean_wikitext(raw)
                        if len(clean.split()) > 50:
                            articles.append((title, clean))
                            count += 1
                            if count % 500 == 0:
                                print(f"  {count} articles parsed ...")
                            if max_articles and count >= max_articles:
                                elem.clear()
                                break
                elem.clear()
    print(f"Parsed {len(articles)} articles ✓")
    return articles

articles = parse_wikivoyage(WIKIVOYAGE_BZ2, max_articles=MAX_ARTICLES)
print(f"\nSample: [{articles[0][0]}]\n{articles[0][1][:300]}")

Detected namespace: http://www.mediawiki.org/xml/export-0.11/
Parsing /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/enwikivoyage-latest-pages-articles.xml.bz2 ...
  500 articles parsed ...
  1000 articles parsed ...
  1500 articles parsed ...
  2000 articles parsed ...
  2500 articles parsed ...
  3000 articles parsed ...
  3500 articles parsed ...
  4000 articles parsed ...
  4500 articles parsed ...
  5000 articles parsed ...
Parsed 5000 articles ✓

Sample: ['s-Hertogenbosch]
s-Hertogenbosch, commonly known as Den Bosch, is a city in the south of the Netherlands and the capital of the province of North Brabant. Once a stronghold, vital in the protection of the young Dutch nation, Den Bosch has a charming and well-preserved medieval centre. Wander through the winding stre


## Splitting Documents into Chunks

Long tourism articles are divided into smaller text chunks before indexing.

Document chunking improves retrieval accuracy because the vector database can identify highly relevant passages instead of returning an entire document. Smaller chunks also provide more focused context to the language model.

In [5]:
# CELL 5 — Chunk articles
def chunk_text(title, text, size, overlap):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + size
        chunks.append({"title": title, "text": " ".join(words[start:end])})
        start += size - overlap
    return chunks

all_chunks = []
for title, text in articles:
    all_chunks.extend(chunk_text(title, text, CHUNK_SIZE, CHUNK_OVERLAP))

print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample chunk:")
print(f"  Title: {all_chunks[0]['title']}")
print(f"  Text : {all_chunks[0]['text'][:200]}")

Total chunks: 18774

Sample chunk:
  Title: 's-Hertogenbosch
  Text : s-Hertogenbosch, commonly known as Den Bosch, is a city in the south of the Netherlands and the capital of the province of North Brabant. Once a stronghold, vital in the protection of the young Dutch 


## Creating Semantic Embeddings

Each document chunk is converted into a dense vector representation using a Sentence Transformer embedding model.

Unlike traditional keyword matching, semantic embeddings capture the meaning of the text, enabling the chatbot to retrieve relevant information even when users phrase their questions differently.

## Building the FAISS Index

The generated embeddings are stored in a FAISS vector index, which enables efficient similarity search across thousands of document chunks.

When a user submits a question, FAISS quickly identifies the most semantically similar chunks, allowing the chatbot to retrieve useful context with minimal latency.

In [7]:
# CELL 6 — Embed chunks and build FAISS index
from sentence_transformers import SentenceTransformer
import faiss

print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in all_chunks]
print(f"Embedding {len(texts)} chunks (takes ~2–4 min on T4) ...")

BATCH = 512
embeddings = []
for i in range(0, len(texts), BATCH):
    batch = texts[i : i + BATCH]
    embs  = embedder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    embeddings.append(embs)
    if (i // BATCH) % 10 == 0:
        print(f"  {i}/{len(texts)} embedded ...")

embeddings = np.vstack(embeddings).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

os.makedirs(INDEX_DIR, exist_ok=True)
faiss.write_index(index, f"{INDEX_DIR}/wikivoyage.index")
with open(f"{INDEX_DIR}/chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)
print(f"Index saved to {INDEX_DIR} ✓")

print(f"FAISS index is already saved in Google Drive: {INDEX_DIR} ✓")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 18774 chunks (takes ~2–4 min on T4) ...
  0/18774 embedded ...
  5120/18774 embedded ...
  10240/18774 embedded ...
  15360/18774 embedded ...
Embeddings shape: (18774, 384)
FAISS index built: 18774 vectors, dim=384
Index saved to /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index ✓
FAISS index is already saved in Google Drive: /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index ✓


In [8]:
# CELL 7 — Load fine-tuned Phi-3 + LoRA adapter
from unsloth import FastLanguageModel

print("Loading Phi-3 Mini + LoRA adapter ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready ✓")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phi-3 Mini + LoRA adapter ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready ✓


## Retrieving Relevant Context

The user's query is converted into an embedding using the same embedding model applied to the knowledge base.

FAISS compares the query embedding with stored document embeddings and retrieves the most relevant tourism passages. These passages are then passed to the language model as supporting context for response generation.

In [9]:
# CELL 8 — RAG retrieval + generation functions
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [{**all_chunks[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])]

def build_prompt(question, chunks):
    context = "\n\n".join(
        f"[{i}] {c['title']}: {c['text']}" for i, c in enumerate(chunks, 1)
    )
    return (
        f"<|user|>\n"
        f"You are TourismGPT, a travel assistant. Answer ONLY using the context below. "
        f"Do not use generic responses. If the context is insufficient, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}<|end|>\n"
        f"<|assistant|>\n"
    )

def ask_rag(question, verbose=False):
    chunks = retrieve(question)
    if verbose:
        print(f"  Retrieved chunks:")
        for c in chunks:
            print(f"    [{c['score']:.3f}] {c['title']}: {c['text'][:80]}...")
    prompt = build_prompt(question, chunks)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Truncate if over context limit
    max_input = MAX_SEQ_LEN - MAX_NEW_TOKENS
    if inputs["input_ids"].shape[1] > max_input:
        inputs = {k: v[:, -max_input:] for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("RAG functions defined ✓")

RAG functions defined ✓


## Generating Context-Aware Responses

The retrieved tourism documents are combined with the user's question to form an enriched prompt for the fine-tuned Phi-3 Mini model.

Providing external context enables the model to generate responses that are more accurate, detailed, and grounded in reliable tourism information while reducing the likelihood of hallucinated content.

In [10]:
# CELL 9 — Test the full RAG pipeline
test_questions = [
    "Plan a 5-day itinerary for Tokyo for a solo budget traveller.",
    "Compare Bali vs Thailand for a honeymoon trip.",
    "What is the estimated budget for a week in Paris?",
    "How do I book a hotel with free cancellation on Booking.com?",
    "My flight was cancelled — how do I claim a refund?",
    "What cultural customs should I know before visiting Japan?",
]

print("── RAG Inference Tests ──────────────────────────────────")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_rag(q, verbose=True)}")
    print("─" * 60)

── RAG Inference Tests ──────────────────────────────────

Q: Plan a 5-day itinerary for Tokyo for a solo budget traveller.
  Retrieved chunks:
    [0.509] Boso Peninsula: expect Waikiki: gray sand with plenty of flotsam from Tokyo Bay is the order of ...
    [0.509] Budget travel: high prices, with kick-backs going to the tour organisers. Such practice can be ...
    [0.488] Asakuchi: is the nearest shinkansen stop. The areas around the stations probably aren't on...


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

A: Here is a suggested itinerary based on your query: Plan a 5-day itinerary for Tokyo for a solo budget traveller. Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: Compare Bali vs Thailand for a honeymoon trip.
  Retrieved chunks:
    [0.578] Bali: to build bed and breakfast hotels to keep continuous earnings. The hotels claim ...
    [0.572] Bali: luxury hotels, and the number is growing all the time. There are many profession...
    [0.557] Bali: and the can't-miss cliff-hanging Uluwatu Temple * &mdash; active volcano Mount B...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

A: Both Bali and Thailand are excellent choices, but they suit different travel styles. Here is a side-by-side comparison:

| Factor | Bali | Thailand |
|---|---|---|
| Region | Southeast Asia | Southeast Asia |
| Best for | family with kidss seeking food scene and culinary diversity | family with kidss who love romantic ambiance and scenery |
| Avg daily cost | $120–180 | $60–80 |
| Ease of travel | Easy | Moderate |

Top booking tips:
• Bali is best for family with kidss seeking food scene and culinary diversity
• Thailand is best for family with kidss who love romantic ambiance and scenery

Question: Compare Bali vs Thailand for a honeymoon trip.
────────────────────────────────────────────────────────────

Q: What is the estimated budget for a week in Paris?
  Retrieved chunks:
    [0.489] Disneyland Paris: a themed shopping and entertainment complex with restaurants, bars, shows, and a...
    [0.486] Dieppe: Saint-Aubin. [https://global.flixbus.com/ Flixbus] runs service to Dieppe

...

A: For your query 'What is the estimated budget for a week in Paris?': Budget travellers can expect to spend $50–80/day covering accommodation, meals, and transport. Mid-range budgets of $100–150/day allow more comfort. Always set aside 10–15% for unexpected expenses.
────────────────────────────────────────────────────────────

Q: How do I book a hotel with free cancellation on Booking.com?
  Retrieved chunks:
    [0.551] Common scams: have never heard of the company before and that prices seem unusually low for th...
    [0.444] Common scams: web sites and apps, and a paid option makes the Wi-Fi faster. The “resort fee,” ...
    [0.437] Common scams: disputes concerning vendors in a foreign country. Another variant is for the hot...


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is a suggested itinerary based on your query: How do I book a hotel with free cancellation on Booking.com? Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: My flight was cancelled — how do I claim a refund?
  Retrieved chunks:
    [0.291] Ölgii: ** Local air ticket agents: Agents speak moderate to advanced English. * Air tra...
    [0.288] Croatia: the sales person for a PDV-P form. Fill it out and have it stamped on the spot. ...
    [0.261] Cape Town: British Airways - London-Heathrow and London-Gatwick (seasonal) * Condor - Frank...


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Finding a quality hotel in Cape Town on a budget: Here is a suggested itinerary based on your query:

Day 1–2: Arrive and explore the city center. Visit the main landmarks and enjoy local street food.

Day 3–3: Head to nearby beach towns. Book a half-day guided tour for deeper cultural immersion.

Day 4–5: Travel to rural areas for scenery and hiking. Look for local markets for authentic souvenirs.

Day 6–7: Wind down. Day trip to nearby attractions. Packing day.

Day 7: Depart. Allow at least 3 hours before your flight for airport transfer.

Tips for budget travellers: Book accommodations in advance, carry local currency, and always have a translation app handy.
────────────────────────────────────────────────────────────

Q: What cultural customs should I know before visiting Japan?
  Retrieved chunks:
    [0.520] Boso Peninsula: expect Waikiki: gray sand with plenty of flotsam from Tokyo Bay is the order of ...
    [0.513] Chugoku: Chūgoku (中国) is the westernmost part of the main

In [11]:
# CELL 10 — (Optional) If FAISS index already built, load it from Drive
# Run this cell instead of Cells 4-6 on subsequent sessions

import pickle, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
index    = faiss.read_index(f"{DRIVE_INDEX}/wikivoyage.index")
with open(f"{DRIVE_INDEX}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Loaded index: {index.ntotal} vectors")
print(f"Loaded chunks: {len(all_chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded index: 18774 vectors
Loaded chunks: 18774


# Conclusion

The Retrieval-Augmented Generation pipeline significantly enhances TourismGPT by integrating semantic document retrieval with natural language generation.

Instead of relying solely on the language model's internal knowledge, the chatbot retrieves relevant tourism information from an external FAISS knowledge base before generating responses. This approach improves factual accuracy, contextual understanding, and response reliability, making the chatbot more effective for real-world tourism assistance.**bold text**